# MNIST — Standard Deep Neural Network (MLP)

Fully connected network on MNIST (28×28 → 256 → 128 → 64 → 10).
No convolutions, no weight sharing.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np, matplotlib.pyplot as plt, time
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_ds = datasets.MNIST('./mnist', train=True, download=True, transform=transform)
test_ds  = datasets.MNIST('./mnist', train=False, download=True, transform=transform)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)
print(f'Train: {len(train_ds)} samples, Test: {len(test_ds)} samples')

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(28*28, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 64)
        self.fc4 = nn.Linear(64, 10)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        return self.fc4(x)

model = MLP().to(device)
print(f'MLP params: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
epochs = 10
train_losses, test_accs = [], []
print('Training MLP...')
for ep in range(epochs):
    model.train(); total_loss = 0; t0 = time.time()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        out = model(x)
        loss = F.cross_entropy(out, y)
        opt.zero_grad(); loss.backward(); opt.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)
    train_losses.append(avg_loss)

    model.eval(); correct = 0; total = 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            correct += (model(x).argmax(1) == y).sum().item()
            total += y.size(0)
    acc = 100 * correct / total
    test_accs.append(acc)
    print(f'  ep {ep+1:2d}  loss={avg_loss:.4f}  acc={acc:.2f}%  {time.time()-t0:.0f}s')

In [ ]:
model.eval(); all_preds, all_labels = [], []
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        all_preds.append(model(x).argmax(1).cpu())
        all_labels.append(y)
all_preds = torch.cat(all_preds).numpy()
all_labels = torch.cat(all_labels).numpy()
cm = confusion_matrix(all_labels, all_preds)
print(f'MLP Final Test Accuracy: {test_accs[-1]:.2f}%')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(train_losses, 'b-o', markersize=3)
axes[0].set_title('MLP Training Loss'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].grid(True)
axes[1].plot(test_accs, 'r-o', markersize=3)
axes[1].set_title('MLP Test Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
axes[1].grid(True)
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay(cm).plot(ax=ax, cmap='Blues', values_format='d')
ax.set_title(f'MLP Confusion Matrix (Acc: {test_accs[-1]:.2f}%)')
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
mis = np.where(all_preds != all_labels)[0][:10]
for i, idx in enumerate(mis):
    img, true_lbl, pred_lbl = test_ds[idx][0].squeeze(), all_labels[idx], all_preds[idx]
    axes[i//5][i%5].imshow(img, cmap='gray')
    axes[i//5][i%5].set_title(f'True:{true_lbl} Pred:{pred_lbl}', fontsize=9, color='red' if true_lbl != pred_lbl else 'green')
    axes[i//5][i%5].axis('off')
plt.suptitle('MLP Misclassified Samples'); plt.tight_layout(); plt.show()